<a href="https://colab.research.google.com/github/majedmb/nmt_md/blob/main/nllb_600M_md_eng_nmt(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 0️⃣  BASIC SETTINGS
# --------------------------------------------------------------
# If you want reproducibility, set a seed.
import random, os, numpy as np, torch
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
os.environ["PYTHONHASHSEED"] = str(seed)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --------------------------------------------------------------
# 1️⃣  INSTALL LIBRARIES
# --------------------------------------------------------------
# Transformers ≥4.35, Datasets, Accelerate, SentencePiece
!pip install -q --upgrade transformers datasets accelerate sentencepiece tqdm


In [ ]:
!pip install datasets
from datasets import load_dataset



In [ ]:
!pip install evaluate
import evaluate
load_metric = evaluate.load('accuracy')

In [ ]:
from accelerate import Accelerator
# Use Accelerate to manage fp16 automatically.
accelerator = Accelerator(mixed_precision='fp16')   # <-- enables mixed precision on GPU

In [ ]:

# --------------------------------------------------------------
# 2️⃣  IMPORTS & ACCELERATOR SETUP
# --------------------------------------------------------------
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)



### Option 1: Uploading a file directly to Colab

This method is suitable for smaller files and will upload the file to the Colab runtime's temporary storage. The files will be deleted once the runtime disconnects.

In [ ]:
from google.colab import files

uploaded = files.upload()

for filename in uploaded.keys():
  print(f'User uploaded file "{filename}" with length {len(uploaded[filename])} bytes')

# Example of how to read a CSV file after uploading:
# import pandas as pd
# df = pd.read_csv(filename)
# display(df.head())

### Option 2: Mounting Google Drive

This method allows you to access files stored in your Google Drive. The files will persist across sessions.

In [ ]:
import pandas as pd

# Replace 'your_file_name.csv' with the actual name of your uploaded CSV file
# If you used 'files.upload()', the file will be in the current directory.
df = pd.read_csv('sentences_cleaned.csv')

# Display the first 5 rows of the DataFrame to confirm it loaded correctly
display(df.head())
dataset = load_dataset("csv", data_files={"train": "sentences_cleaned.csv"})
print(dataset)
# Expected splits: 'train' and 'validation'
# Columns: 'darija' (source) and 'english' (target)

In [ ]:
# Check for null values in 'darija' and 'eng' columns in the original DataFrame
print("Null values in 'darija' column:", df['darija'].isnull().sum())
print("Null values in 'eng' column:", df['eng'].isnull().sum())

# Filter out rows where 'darija' or 'eng' columns are None from the dataset
# This ensures that all examples passed to the tokenizer are valid strings.
dataset["train"] = dataset["train"].filter(lambda example: example["darija"] is not None and example["eng"] is not None)

In [ ]:
# --------------------------------------------------------------
# 4️⃣  TOKENIZER & MODEL
# --------------------------------------------------------------
# NLLB‑200 Distilled‑600M (multilingual, 200 langs).
# Language codes: darija = "mos" (Moroccan Arabic), english = "eng_Latn".
model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

# Load the model in "seq2seq" mode.
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
# --------------------------------------------------------------
# 5️⃣  PRE‑PROCESSING FUNCTION
# --------------------------------------------------------------
src_lang = "mos"          # Darija language code in NLLB md as source
tgt_lang = "eng_Latn"    # English language code

max_source_length = 128   # Adjusted to save memory
max_target_length = 128   # Adjusted to save memory

def preprocess_function(examples):
    # Prefix the language token that NLLB expects.
    inputs = [
        f"{src_lang} " + src for src in examples["darija"]
    ]
    targets = [
        f"{tgt_lang} " + tgt for tgt in examples["eng"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=max_source_length,
        truncation=True,
        padding=False,           # padding will be done later by the collator
    )
    # Tokenize targets directly
    labels = tokenizer(
        targets,
        max_length=max_target_length,
        truncation=True,
        padding=False,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
# Apply preprocessing (this creates tokenized columns)
tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

In [ ]:
# --------------------------------------------------------------
# 6️⃣  DATA COLLATOR (dynamic padding)
# --------------------------------------------------------------
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    padding="longest",   # pad to longest sequence in the batch
    label_pad_token_id=-100,  # ignore index for loss computation
    return_tensors="pt", #for memory optimization
    pad_to_multiple_of=8, #for memory optimization
)



In [ ]:
!pip install sacrebleu


In [ ]:
# --------------------------------------------------------------
# 7️⃣  METRICS (BLEU for translation quality)
# --------------------------------------------------------------
bleu = evaluate.load("sacrebleu")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    # Replace -100 (ignore index) with pad token id for decoding
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # For labels, shift left (remove the initial BOS token) and decode
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # SacreBLEU expects a list of references per hypothesis
    bleu_score = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    return {"bleu": bleu_score["score"]}

### 6️⃣ Training Setup: Data Collator & Training Arguments

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Define training arguments
training_args = Seq2SeqTrainingArguments(
    #output_dir="./nllb-finetuned-doda",  # directory to store checkpoints
    output_dir="/content/drive/MyDrive/model_fine_tuned_nllb600m/model_md_eng_nmt/checkpoints",  # directory to store checkpoints persisted

    #evaluation_strategy="epoch",
    save_strategy="epoch",  #save checkpoint after every epoch

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,  # limit the number of checkpoints
    num_train_epochs=4,
    predict_with_generate=True,
    fp16=True, # enable fp16 training #disable if no GPU
    optim="adafactor", # use Adafactor optimizer
    logging_dir="./logs",
    push_to_hub=False, # set to True to push to Hugging Face Hub
)

### 7️⃣ Define Metrics

In [ ]:
metric = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

In [ ]:
!pip install transformers --upgrade

In [ ]:
output_dir = "/content/drive/MyDrive/model_fine_tuned_nllb600m/model_md_eng_nmt/checkpoints"

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    # evaluation_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=4,      # Further reduced to 1
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=16,     # effective batch size = 16
    num_train_epochs=4,
    save_total_limit=4,  # limit the number of checkpoints
    weight_decay=0.01,
    fp16=True,
    logging_dir=f"{output_dir}/logs",
    logging_steps=10,
    report_to="none",
    predict_with_generate=True,
    generation_max_length=max_target_length,
    optim="adafactor",                 # Use Adafactor instead of AdamW
    save_strategy="epoch", #save checkpoint after every epoch
    resume_from_checkpoint=True #start training from checkpoints
)

In [ ]:
# --------------------------------------------------------------
# 8️⃣  TRAINING ARGUMENTS (Optimized for Memory)
# --------------------------------------------------------------
output_dir = "/content/drive/MyDrive/model_fine_tuned_nllb600m/model_md_eng_nmt/model2"

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    # evaluation_strategy="epoch",
    save_strategy="epoch",
    #learning_rate=3e-5,
    #per_device_train_batch_size=8,      # Further reduced to 1
    #per_device_eval_batch_size=8,
    per_device_train_batch_size=4,      # Further reduced to 1
    per_device_eval_batch_size=4,
    #gradient_accumulation_steps=16,     # effective batch size = 16
    gradient_accumulation_steps=2,     # effective batch size = 16
    learning_rate=2e-5,


    num_train_epochs=5,
    #num_train_epochs=4,
    save_total_limit=3,  # limit the number of checkpoints
    #weight_decay=0.01,
    weight_decay=0.05,

    fp16=True,
    logging_dir=f"{output_dir}/logs",
    logging_steps=10,
    report_to="none",
    predict_with_generate=True,
    generation_max_length=max_target_length,
    optim="adafactor",                 # Use Adafactor instead of AdamW
)

In [ ]:
!PYTORCH_ALLOC_CONF=expandable_segments:True

### 6️⃣ Create Validation Split

In [ ]:
if 'validation' not in tokenized_datasets:
    tokenized_datasets = tokenized_datasets['train'].train_test_split(test_size=0.1, seed=seed)
    tokenized_datasets['validation'] = tokenized_datasets['test']
    del tokenized_datasets['test']
print(tokenized_datasets)

### 8️⃣ Initialize and Train the Model

In [ ]:
checkpoint_path = "/content/drive/MyDrive/model_fine_tuned_nllb600m/model_md_eng_nmt/model/checkpoint-688"
trainer = Seq2SeqTrainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Option 1: Start fresh since the weights are missing in the checkpoint folder
trainer.train()

# Option 2: Try the older checkpoint if it has the weights
#trainer.train(resume_from_checkpoint = checkpoint_path)

**Save the trained model**

In [ ]:
import os
from google.colab import drive

# 1. Define the path
drive_path = "/content/drive/MyDrive/model_fine_tuned_nllb600m/model_md_eng_nmt/model2/model"
os.makedirs(drive_path, exist_ok=True)

# 2. Save the model and tokenizer
print(f"Starting save to: {drive_path}...")
trainer.save_model(drive_path)
tokenizer.save_pretrained(drive_path)

# 3. FORCE PERSISTENCE
# This ensures the buffers are flushed to Google Drive's backend
print("Forcing persistence to Google Drive (this may take a moment)...")
drive.flush_and_unmount()

# 4. Remount to confirm files are there
drive.mount('/content/drive')

if os.path.exists(os.path.join(drive_path, 'pytorch_model.bin')) or os.path.exists(os.path.join(drive_path, 'model.safetensors')):
    print(f"\n✅ SUCCESS: Model weights confirmed at {drive_path}")
else:
    print("\n❌ WARNING: Weights still not detected. Check Drive sync status.")

In [ ]:
# --------------------------------------------------------------
# 8​​​​​​  TRAINING ARGUMENTS (Optimized for Memory 2)
# --------------------------------------------------------------
output_dir = "/content/drive/MyDrive/model_fine_tuned_nllb600m/model_md_eng_nmt/model2/checkpoint"

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    # BATCH performance
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    # GPU speed
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    fp16=True,

    # optimizer & scheduler
    learning_rate=3e-5,
    lr_scheduler_type="linear",
    warmup_ratio=0.05,
    weight_decay=0.01,

    # Training control
    num_train_epochs=4,
    logging_steps=100,

    # Strategy Sync: Both MUST be 'epoch' for load_best_model_at_end
    eval_strategy="epoch",      # Added to fix the ValueError
    save_strategy="epoch",

    # Best model saved
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,

    # For Translation
    predict_with_generate=True,
    generation_max_length=128,

    # Stability
    label_smoothing_factor=0.1,
    optim="adafactor",

    # Memory optimization
    # group_by_length=True,        # Set manually below to bypass version-specific TypeError

    # Logging
    report_to="none",

    # Reproducibility
    seed=42,
)

# Manually setting group_by_length for memory optimization to bypass constructor validation
training_args.group_by_length = True
print(f"group_by_length is enabled: {training_args.group_by_length}")

In [ ]:
import os

checkpoint_dir = "/content/drive/MyDrive/model_fine_tuned_nllb600m/model_md_eng_nmt/checkpoints"

if os.path.exists(checkpoint_dir):
    print(f"Contenu de {checkpoint_dir} :")
    content = os.listdir(checkpoint_dir)
    for item in content:
        print(f" - {item}")
else:
    print(f"Le chemin {checkpoint_dir} n'existe pas. Vérifiez le montage du Drive.")

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
import os

# Define the path in Google Drive where you want to save the model
drive_path = "/content/drive/MyDrive/model_fine_tuned_nllb600m/model_md_eng_nmt/model2"

# Create the directory if it doesn't exist
os.makedirs(drive_path, exist_ok=True)

# Save the model and tokenizer
trainer.save_model(drive_path)
tokenizer.save_pretrained(drive_path)

print(f"Model and tokenizer saved to: {drive_path}")

In [ ]:
!pip install -U bitsandbytes

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
import torch
import os

# Chemin vers votre modèle sur Google Drive
drive_path = "/content/drive/MyDrive/nllb-finetuned-darija-model2"

if not os.path.exists(drive_path):
    print(f"Erreur : Le dossier {drive_path} n'existe pas. Vérifiez que votre Drive est monté.")
else:
    # Configuration pour le chargement en 8-bit
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)

    # Chargement en spécifiant local_files_only=True pour éviter l'erreur OSError du Hub
    print("Chargement du tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(drive_path, local_files_only=True)

    print("Chargement du modèle (8-bit)...")
    model = AutoModelForSeq2SeqLM.from_pretrained(
        drive_path,
        quantization_config=bnb_config,
        local_files_only=True
    )

    model.eval()
    print("✅ Modèle et tokenizer chargés avec succès depuis Google Drive.")

In [ ]:
def translate_darija_to_english(text):
    # 1. Configure tokenizer languages correctly
    tokenizer.src_lang = "mos"
    tokenizer.tgt_lang = "eng_Latn"

    # 2. Tokenize input
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    # 3. Get the correct forced BOS token ID for NLLB-200
    # Note: NLLB-200 usually expects double underscores for language tokens
    forced_bos_token_id = tokenizer.convert_tokens_to_ids("__eng_Latn__")

    # 4. Generate with explicit parameters
    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        max_new_tokens=128,
        max_length=None
    )

    # 5. Decode
    translation = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

    return translation.strip()

# Test again
darija_sentence = "Homa mkhbbyin chi haja, ana mti99en!"
english_translation = translate_darija_to_english(darija_sentence)

print(f"Darija: {darija_sentence}")
print(f"English (Translated): {english_translation}")

In [ ]:
import os
# List contents of Drive to find the correct model folder
drive_root = "/content/drive/MyDrive/"
if os.path.exists(drive_root):
    print("Contenu de votre Google Drive :")
    content = os.listdir(drive_root)
    for item in content:
        if "nllb" in item.lower() or "darija" in item.lower():
            print(f"[TROUVÉ] {item}")
        else:
            print(f" - {item}")
else:
    print("Le Drive ne semble pas être monté à /content/drive/MyDrive/")

In [ ]:
def translate_darija_to_english(text):
    # Ensure the correct source and target language tokens are used for generation
    # The model was fine-tuned for 'mos' (Darija) as source and 'eng_Latn' (English) as target
    # So, we set the source_lang for the tokenizer input and target_lang for generation
    tokenizer.src_lang = "mos"

    # Prepend the target language token to the input for generation (NLLB specific)
    input_ids = tokenizer(f"{text}", return_tensors="pt", padding=True, truncation=True).to(model.device) # Use model.device

    # Generate translation, specifying the target language
    generated_tokens = model.generate(
        **input_ids,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("__eng_Latn__"), # Correct method for NLLB
        max_new_tokens=128
    )

    # Decode the generated tokens
    translation = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
    return translation

# Example usage:
darija_sentence = "Ana safi ghadi nahawel tani" #"Homa mkhbbyin chi haja, ana mti99en!"
english_translation = translate_darija_to_english(darija_sentence)


print(f"Darija: {darija_sentence}")
print(f"English (Translated): {english_translation}")

In [ ]:
import torch

# Check if model and tokenizer exist in the global scope
model_exists = 'model' in globals()
tokenizer_exists = 'tokenizer' in globals()

print(f"Modèle en mémoire : {model_exists}")
print(f"Tokenizer en mémoire : {tokenizer_exists}")

if model_exists:
    print(f"Type du modèle : {type(model)}")
    try:
        print(f"Appareil (Device) : {model.device}")
    except:
        pass

if tokenizer_exists:
    print(f"Type du tokenizer : {type(tokenizer)}")

### Note on Translation Direction

As explained, this fine-tuned model is optimized for **Darija to English** translation because of how the `src_lang` and `tgt_lang` were configured during preprocessing and training. If you need to translate from **English to Darija**, you would typically need to:

1.  **Use the base NLLB model (not fine-tuned)** and configure its `source_lang` to `eng_Latn` and `target_lang` to `mos` for inference.
2.  **Fine-tune another NLLB model instance** specifically for English to Darija, reversing the source and target language settings during the training process.

You can verify that the model files are saved in your Google Drive by checking the specified path in your Google Drive folder structure.

In [ ]:
def print_model_size(model):
    param_size = 0
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()

    size_all_mb = (param_size + buffer_size) / 1024**2
    print('Nombre de paramètres : {:,}'.format(sum(p.numel() for p in model.parameters())))
    print('Taille totale du modèle : {:.2f} Mo'.format(size_all_mb))
    print('Mémoire estimée avec optimizer (Adam) : {:.2f} Mo'.format(size_all_mb * 4)) # Adam stores 2 extra states per param

print_model_size(model)

In [ ]:
import transformers
print(f"Transformers version: {transformers.__version__}")

# Task
Translate additional Darija sentences to English using the fine-tuned NLLB model, thoroughly review the "sentences.csv" dataset for data quality issues, and consider adjusting the training parameters (e.g., `num_train_epochs`, `learning_rate`, `per_device_train_batch_size`, `gradient_accumulation_steps`) in cell `bnErtdBEf3LT` to identify the most impactful next steps for model improvement.

## Translate Multiple Examples

### Subtask:
Translate a diverse set of Darija sentences to English using the fine-tuned model to understand translation quality and identify potential errors.


**Reasoning**:
To fulfill the subtask instructions, I will define a list of Darija sentences and then iterate through them, calling the `translate_darija_to_english` function for each and printing the results.



In [ ]:
darija_sentences_to_translate = [
    "Salam, kif dayer?",
    "Fin ghadi?",
    "Hadchi zwin bezaf.",
    "Baghi nsafer l Marrakech.",
    "Shukran bezaf 3la l'mosa3ada.",
    "Wash kayn shi makan hna feen nakol?"
]

print("\n--- Translating Multiple Examples ---")
for sentence in darija_sentences_to_translate:
    english_translation = translate_darija_to_english(sentence)
    print(f"Darija: {sentence}")
    print(f"English (Translated): {english_translation}\n")


**Reasoning**:
The previous output showed a warning about `max_new_tokens` and `max_length` in the `generate` function. To address this, I will modify the `translate_darija_to_english` function in cell `803fe48e` to explicitly remove `max_length` from the `model.generate` call, as `max_new_tokens` is already set and preferred. This will silence the warning without changing the intended behavior. After that, I'll translate more sentences to comply with the subtask to continue with additional sentences to further assess the model's performance.



In [ ]:
def translate_darija_to_english(text):
    # Ensure the correct source and target language tokens are used for generation
    # The model was fine-tuned for 'mos' (Darija) as source and 'eng_Latn' (English) as target
    # So, we set the source_lang for the tokenizer input and target_lang for generation
    tokenizer.src_lang = "mos"

    # Prepend the target language token to the input for generation (NLLB specific)
    input_ids = tokenizer(f"{text}", return_tensors="pt", padding=True, truncation=True).to(model.device) # Use model.device

    # Generate translation, specifying the target language
    generated_tokens = model.generate(
        **input_ids,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("__eng_Latn__"), # Correct method for NLLB
        max_new_tokens=128
    )

    # Decode the generated tokens
    translation = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
    return translation

# Example usage:
darija_sentence = "Ana safi ghadi nahawel tani" #"Homa mkhbbyin chi haja, ana mti99en!"
english_translation = translate_darija_to_english(darija_sentence)


print(f"Darija: {darija_sentence}")
print(f"English (Translated): {english_translation}")

**Reasoning**:
The previous output showed a warning about `max_new_tokens` and `max_length` in the `generate` function. Although `max_length` is not explicitly set in the function, it's implicitly derived from the model's configuration. To explicitly resolve this and silence the warning, I will modify the `translate_darija_to_english` function by setting `max_length=None` in the `model.generate` call, which ensures `max_new_tokens` is the sole limit for generation length without conflict. Then, I will provide a new set of sentences for translation to continue assessing the model's performance as per the subtask.



In [ ]:
def translate_darija_to_english(text):
    # Ensure the correct source and target language tokens are used for generation
    # The model was fine-tuned for 'mos' (Darija) as source and 'eng_Latn' (English) as target
    # So, we set the source_lang for the tokenizer input and target_lang for generation
    tokenizer.src_lang = "mos"

    # Prepend the target language token to the input for generation (NLLB specific)
    input_ids = tokenizer(f"{text}", return_tensors="pt", padding=True, truncation=True).to(model.device) # Use model.device

    # Generate translation, specifying the target language
    generated_tokens = model.generate(
        **input_ids,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("__eng_Latn__"), # Correct method for NLLB
        max_new_tokens=128,
        max_length=None # Explicitly set to None to avoid warning when max_new_tokens is set
    )

    # Decode the generated tokens
    translation = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
    return translation

# Example usage:
darija_sentence = "Ana safi ghadi nahawel tani" #"Homa mkhbbyin chi haja, ana mti99en!"
english_translation = translate_darija_to_english(darija_sentence)


print(f"Darija: {darija_sentence}")
print(f"English (Translated): {english_translation}")

**Reasoning**:
The previous steps updated the `translate_darija_to_english` function and removed a warning. Now, to further assess the model's performance as per the subtask, I will define a new list of Darija sentences and translate them using the updated function.



In [ ]:
darija_sentences_to_translate = [
    "Salam, kif dayer?",
    "Fin ghadi?",
    "Hadchi zwin bezaf.",
    "Baghi nsafer l Marrakech.",
    "Shukran bezaf 3la l'mosa3ada.",
    "Wash kayn shi makan hna feen nakol?",
    "Ana bghit nchouf l film jdid.",
    "Kifash nqder nweqef lkar.",
    "Had l'makla bnina bezaf.",
    "Ashmn weqt radi tji?"
]

print("\n--- Translating Multiple Examples (Updated) ---")
for sentence in darija_sentences_to_translate:
    english_translation = translate_darija_to_english(sentence)
    print(f"Darija: {sentence}")
    print(f"English (Translated): {english_translation}\n")

**Reasoning**:
Upon reviewing the `sentences.csv` dataset, a significant data quality issue was identified: approximately 75,000 out of 87,000 English translations were null. This severely reduced the effective training dataset size to only around 11,000 examples after filtering. Given this limitation, one of the most impactful next steps for model improvement is to allow the model to train for more epochs, ensuring it makes the most of the limited clean data available. Therefore, I will adjust the `num_train_epochs` in the training arguments.



In [ ]:
# --------------------------------------------------------------
# 8️⃣  TRAINING ARGUMENTS (Optimized for Memory)
# --------------------------------------------------------------
output_dir = "/content/nllb-darija-finetuned"

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    # evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=4,      # Further reduced to 1
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=16,     # effective batch size = 16
    num_train_epochs=8, # Increased epochs to make better use of limited clean data
    weight_decay=0.01,
    fp16=True,
    logging_dir=f"{output_dir}/logs",
    logging_steps=10,
    report_to="none",
    predict_with_generate=True,
    generation_max_length=max_target_length,
    optim="adafactor",                 # Use Adafactor instead of AdamW
)

**Reasoning**:
The previous step produced a deprecation warning about `logging_dir`. To address this, I will remove the `logging_dir` parameter from the `Seq2SeqTrainingArguments` as `report_to="none"` indicates no logging to external platforms is desired, and setting this parameter is no longer necessary or recommended.



In [ ]:
import os
from transformers import Seq2SeqTrainingArguments

# Define the output directory directly in Google Drive to persist checkpoints
output_dir = "/content/drive/MyDrive/nllb-darija-checkpoints"
os.makedirs(output_dir, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    save_strategy="epoch",           # Save a checkpoint at the end of every epoch
    save_total_limit=3,              # Keep only the last 3 checkpoints to save space
    learning_rate=3e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=64,
    num_train_epochs=8,
    weight_decay=0.01,
    fp16=True,
    logging_steps=10,
    report_to="none",
    predict_with_generate=True,
    generation_max_length=max_target_length,
    optim="adafactor",
)

**Reasoning**:
The previous step produced a deprecation warning about `logging_dir`. To address this, I will remove the `logging_dir` parameter from the `Seq2SeqTrainingArguments` as `report_to="none"` indicates no logging to external platforms is desired, and setting this parameter is no longer necessary or recommended. The `num_train_epochs` will remain at 8 as intended from the previous step.



In [ ]:
output_dir = "/content/nllb-darija-finetuned"

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    # evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=16,
    num_train_epochs=8,
    weight_decay=0.01,
    fp16=True,
    # logging_dir=f"{output_dir}/logs", # Removed to address deprecation warning
    logging_steps=10,
    report_to="none",
    predict_with_generate=True,
    generation_max_length=max_target_length,
    optim="adafactor",
)

**Reasoning**:
The training arguments have been updated. Now, I need to re-initialize the `Seq2SeqTrainer` with these new `training_args` to ensure the model trains with the increased number of epochs and without the `logging_dir` warning. After re-initialization, I will restart the training process.



In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# To resume from a checkpoint if training stopped, set resume_from_checkpoint=True
# trainer.train(resume_from_checkpoint=True)
trainer.train()

**Reasoning**:
The error `ValueError: You cannot perform fine-tuning on purely quantized models` indicates that the model was loaded with 8-bit quantization in a previous step, which prevents direct fine-tuning. To resolve this, I need to reload the fine-tuned model saved in Google Drive in full precision (without quantization) before re-initializing the `Seq2SeqTrainer`.



In [ ]:
drive_path = "/content/drive/MyDrive/nllb-finetuned-darija-model"

# Reload the model and tokenizer in full precision for fine-tuning
tokenizer = AutoTokenizer.from_pretrained(drive_path)
model = AutoModelForSeq2SeqLM.from_pretrained(drive_path)

trainer = Seq2SeqTrainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"], # Assuming 'validation' split exists
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()

**Reasoning**:
The previous step resulted in a `OutOfMemoryError` due to reloading the model in full precision for training. To mitigate this, I will further reduce the `per_device_train_batch_size` to 1 and increase `gradient_accumulation_steps` to 64 in the `Seq2SeqTrainingArguments` to maintain an effective batch size while minimizing memory usage. I will also explicitly set `fp16=True` to ensure mixed precision training is utilized for memory efficiency.



In [ ]:
output_dir = "/content/nllb-darija-finetuned"

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    # evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=1,      # Further reduced to 1
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=64,     # effective batch size = 64 (1 * 64)
    num_train_epochs=8,
    weight_decay=0.01,
    fp16=True,
    # logging_dir=f"{output_dir}/logs", # Removed to address deprecation warning
    logging_steps=10,
    report_to="none",
    predict_with_generate=True,
    generation_max_length=max_target_length,
    optim="adafactor",
)

**Reasoning**:
The training arguments have been updated to further optimize memory usage. Now, I need to re-initialize the `Seq2SeqTrainer` with these new `training_args` and then restart the training process to attempt to resolve the `OutOfMemoryError`.



In [ ]:
trainer = Seq2SeqTrainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"], # Assuming 'validation' split exists
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()

**Reasoning**:
The model is still encountering an `OutOfMemoryError` even after reducing batch size and increasing gradient accumulation steps. This indicates that the full precision model is too large for the available GPU memory. To overcome this, I will re-enable 8-bit quantization for the base model and use Parameter-Efficient Fine-Tuning (PEFT) with LoRA, which allows training large models with significantly less memory by only training small adapter layers.



In [ ]:
import bitsandbytes
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Re-install bitsandbytes if not already installed (to ensure it's available for 8-bit)
try:
    import bitsandbytes
except ImportError:
    !pip install bitsandbytes
    import bitsandbytes

# Configure for 8-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
)

# Reload the base NLLB model with 8-bit quantization
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, quantization_config=bnb_config)

# Prepare the model for k-bit training (required for PEFT on quantized models)
model = prepare_model_for_kbit_training(model)

# Configure LoRA
lora_config = LoraConfig(
    r=8, # LoRA attention dimension
    lora_alpha=32, # Alpha parameter for LoRA scaling
    target_modules=["query_proj", "value_proj"], # Target specific attention projection layers
    lora_dropout=0.05, # Dropout probability for LoRA layers
    bias="none", # Bias type for LoRA layers
    task_type="SEQ_2_SEQ_LM" # Task type for sequence-to-sequence language modeling
)

# Get the PEFT model
model = get_peft_model(model, lora_config)

# Print trainable parameters to verify PEFT setup
model.print_trainable_parameters()

# Re-initialize the trainer with the PEFT model
trainer = Seq2SeqTrainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"], # Assuming 'validation' split exists
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()

**Reasoning**:
The model is still encountering an `OutOfMemoryError` during loading, even with 8-bit quantization and PEFT. This indicates that even 8-bit is too memory-intensive. To further reduce memory footprint during model loading and training, I will switch to 4-bit quantization using `load_in_4bit=True` in `BitsAndBytesConfig` and apply PEFT with LoRA.



In [ ]:
import bitsandbytes
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

# Configure for 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # Use 4-bit quantization
    bnb_4bit_quant_type="nf4", # Use NormalFloat4 quantization
    bnb_4bit_compute_dtype=torch.float16, # Use float16 for computation
)

# Reload the base NLLB model with 4-bit quantization
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, quantization_config=bnb_config)

# Prepare the model for k-bit training (required for PEFT on quantized models)
model = prepare_model_for_kbit_training(model)

# Configure LoRA
lora_config = LoraConfig(
    r=8, # LoRA attention dimension
    lora_alpha=32, # Alpha parameter for LoRA scaling
    target_modules=["query_proj", "value_proj"], # Target specific attention projection layers
    lora_dropout=0.05, # Dropout probability for LoRA layers
    bias="none", # Bias type for LoRA layers
    task_type="SEQ_2_SEQ_LM" # Task type for sequence-to-sequence language modeling
)

# Get the PEFT model
model = get_peft_model(model, lora_config)

# Print trainable parameters to verify PEFT setup
model.print_trainable_parameters()

# Re-initialize the trainer with the PEFT model
trainer = Seq2SeqTrainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"], # Assuming 'validation' split exists
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()

**Reasoning**:
The model is still encountering an `OutOfMemoryError` during loading, even with 4-bit quantization and PEFT. This suggests that the model, even when quantized, cannot fully fit into GPU memory. To resolve this, I will explicitly set `device_map="auto"` when loading the model to offload parts of it to CPU RAM if necessary, enabling the model to load successfully and proceed with PEFT training.



In [ ]:
import bitsandbytes
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

# Configure for 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # Use 4-bit quantization
    bnb_4bit_quant_type="nf4", # Use NormalFloat4 quantization
    bnb_4bit_compute_dtype=torch.float16, # Use float16 for computation
)

# Reload the base NLLB model with 4-bit quantization
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForSeq22SeqLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto") # Added device_map="auto"

# Prepare the model for k-bit training (required for PEFT on quantized models)
model = prepare_model_for_kbit_training(model)

# Configure LoRA
lora_config = LoraConfig(
    r=8, # LoRA attention dimension
    lora_alpha=32, # Alpha parameter for LoRA scaling
    target_modules=["query_proj", "value_proj"], # Target specific attention projection layers
    lora_dropout=0.05, # Dropout probability for LoRA layers
    bias="none", # Bias type for LoRA layers
    task_type="SEQ_2_SEQ_LM" # Task type for sequence-to-sequence language modeling
)

# Get the PEFT model
model = get_peft_model(model, lora_config)

# Print trainable parameters to verify PEFT setup
model.print_trainable_parameters()

# Re-initialize the trainer with the PEFT model
trainer = Seq2SeqTrainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"], # Assuming 'validation' split exists
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()


**Reasoning**:
The previous code failed due to a `NameError` caused by a typo in the class name `AutoModelForSeq22SeqLM`. I will correct this to `AutoModelForSeq2SeqLM` to properly load the model.



In [ ]:
import bitsandbytes
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

# Configure for 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # Use 4-bit quantization
    bnb_4bit_quant_type="nf4", # Use NormalFloat4 quantization
    bnb_4bit_compute_dtype=torch.float16, # Use float16 for computation
)

# Reload the base NLLB model with 4-bit quantization
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto") # Corrected typo: AutoModelForSeq2SeqLM

# Prepare the model for k-bit training (required for PEFT on quantized models)
model = prepare_model_for_kbit_training(model)

# Configure LoRA
lora_config = LoraConfig(
    r=8, # LoRA attention dimension
    lora_alpha=32, # Alpha parameter for LoRA scaling
    target_modules=["query_proj", "value_proj"], # Target specific attention projection layers
    lora_dropout=0.05, # Dropout probability for LoRA layers
    bias="none", # Bias type for LoRA layers
    task_type="SEQ_2_SEQ_LM" # Task type for sequence-to-sequence language modeling
)

# Get the PEFT model
model = get_peft_model(model, lora_config)

# Print trainable parameters to verify PEFT setup
model.print_trainable_parameters()

# Re-initialize the trainer with the PEFT model
trainer = Seq2SeqTrainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"], # Assuming 'validation' split exists
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()

**Reasoning**:
The previous error indicated that the `target_modules` (`'query_proj'`, `'value_proj'`) specified in the `LoraConfig` were not found in the NLLB model. I will correct these to the more common and likely names for query and value projection layers in transformer models, which are `'q_proj'` and `'v_proj'`. This change should allow PEFT to correctly identify and attach LoRA adapters.



In [ ]:
import bitsandbytes
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

# Configure for 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # Use 4-bit quantization
    bnb_4bit_quant_type="nf4", # Use NormalFloat4 quantization
    bnb_4bit_compute_dtype=torch.float16, # Use float16 for computation
)

# Reload the base NLLB model with 4-bit quantization
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto") # Corrected typo: AutoModelForSeq2SeqLM

# Prepare the model for k-bit training (required for PEFT on quantized models)
model = prepare_model_for_kbit_training(model)

# Configure LoRA
lora_config = LoraConfig(
    r=8, # LoRA attention dimension
    lora_alpha=32, # Alpha parameter for LoRA scaling
    target_modules=["q_proj", "v_proj"], # Corrected target modules
    lora_dropout=0.05, # Dropout probability for LoRA layers
    bias="none", # Bias type for LoRA layers
    task_type="SEQ_2_SEQ_LM" # Task type for sequence-to-sequence language modeling
)

# Get the PEFT model
model = get_peft_model(model, lora_config)

# Print trainable parameters to verify PEFT setup
model.print_trainable_parameters()

# Re-initialize the trainer with the PEFT model
trainer = Seq2SeqTrainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"], # Assuming 'validation' split exists
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()

## Translate Multiple Examples with PEFT-tuned Model

### Subtask:
Translate a diverse set of Darija sentences to English using the newly fine-tuned PEFT model to assess the impact of PEFT training and compare with previous results.


**Reasoning**:
I need to save the PEFT adapters of the trained model to a specified directory in Google Drive. This will ensure that the fine-tuned adapters are preserved for later use and can be reloaded for inference.



In [ ]:
import os

# Define the path in Google Drive where you want to save the PEFT adapters
peft_drive_path = "/content/drive/MyDrive/nllb-finetuned-darija-model-peft"

# Create the directory if it doesn't exist
os.makedirs(peft_drive_path, exist_ok=True)

# Save only the PEFT adapters
trainer.model.save_pretrained(peft_drive_path)

print(f"PEFT adapters saved to: {peft_drive_path}")